In [1]:
# Bloco 1 - Importar a base de rotas com cidade e estado

from pathlib import Path
import pandas as pd


def obter_pasta_do_notebook():
    return Path.cwd()


def importar_csv_mesma_pasta(nome_arquivo, nome_base, colunas=None, sep=",", engine=None):
    try:
        pasta_notebook = obter_pasta_do_notebook()
        caminho_arquivo = pasta_notebook / nome_arquivo

        argumentos = {
            "filepath_or_buffer": caminho_arquivo,
            "sep": sep,
            "encoding": "utf-8-sig",
            "usecols": colunas
        }

        if engine is not None:
            argumentos["engine"] = engine
        else:
            argumentos["low_memory"] = False

        df = pd.read_csv(**argumentos)
        df.columns = df.columns.str.strip()

        print(f"Base '{nome_base}' importada com sucesso!")
        print(f"Arquivo importado: {caminho_arquivo}")
        print(f"Quantidade de linhas: {df.shape[0]}")
        print(f"Quantidade de colunas: {df.shape[1]}")
        print()

        return df

    except FileNotFoundError:
        print(f"Erro: arquivo da base '{nome_base}' não encontrado.")
        print(f"Verifique se o arquivo '{nome_arquivo}' está na mesma pasta do notebook.")
        return None

    except Exception as erro:
        print(f"Erro ao importar a base '{nome_base}'.")
        print(f"Detalhes do erro: {erro}")
        return None


def inserir_cidade_estado(df_rotas, df_municipios):
    df_rotas = df_rotas.copy()
    df_municipios = df_municipios.copy()

    df_rotas.columns = df_rotas.columns.str.strip()
    df_municipios.columns = df_municipios.columns.str.strip()

    colunas_municipios_necessarias = [
        "Código Município Completo",
        "Nome_Município",
        "Nome_UF"
    ]

    colunas_faltantes = [
        coluna for coluna in colunas_municipios_necessarias
        if coluna not in df_municipios.columns
    ]

    if colunas_faltantes:
        print("Erro: algumas colunas necessárias não foram encontradas na base de municípios.")
        print("Colunas faltantes:", colunas_faltantes)
        return None

    df_rotas["codigo_cidade"] = pd.to_numeric(
        df_rotas["codigo_cidade"],
        errors="coerce"
    ).astype("Int64")

    df_municipios["Código Município Completo"] = pd.to_numeric(
        df_municipios["Código Município Completo"],
        errors="coerce"
    ).astype("Int64")

    df_municipios_resumido = (
        df_municipios[
            [
                "Código Município Completo",
                "Nome_Município",
                "Nome_UF"
            ]
        ]
        .drop_duplicates(subset=["Código Município Completo"], keep="first")
        .rename(
            columns={
                "Código Município Completo": "codigo_cidade",
                "Nome_Município": "nome_cidade",
                "Nome_UF": "estado"
            }
        )
    )

    df_rotas_transformado = df_rotas.merge(
        df_municipios_resumido,
        on="codigo_cidade",
        how="left"
    )

    colunas = list(df_rotas_transformado.columns)
    colunas.remove("nome_cidade")
    colunas.remove("estado")

    posicao_codigo_cidade = colunas.index("codigo_cidade")
    colunas.insert(posicao_codigo_cidade + 1, "nome_cidade")
    colunas.insert(posicao_codigo_cidade + 2, "estado")

    df_rotas_transformado = df_rotas_transformado[colunas]

    print("Dados transformados com sucesso!")
    print(
        "Quantidade de rotas sem cidade/estado encontradas:",
        df_rotas_transformado["nome_cidade"].isna().sum()
    )

    return df_rotas_transformado


nome_arquivo_rotas = "script_03_rotas.csv"
nome_arquivo_municipios = "Lista de Municipios IBGE.csv"

colunas_rotas = [
    "codigo_cidade",
    "id_rota",
    "da_porteira",
    "da_mataburro",
    "da_colchete",
    "da_atoleiro",
    "da_ponterustica",
    "tipo",
    "shape"
]

df_rotas = importar_csv_mesma_pasta(
    nome_arquivo_rotas,
    "rotas",
    colunas=colunas_rotas,
    sep="," 
)

df_municipios = importar_csv_mesma_pasta(
    nome_arquivo_municipios,
    "municípios",
    sep=None,
    engine="python"
)

if df_rotas is not None and df_municipios is not None:
    df_rotas_transformado = inserir_cidade_estado(df_rotas, df_municipios)

    if df_rotas_transformado is not None:
        display(df_rotas_transformado)

        # Exporta o DataFrame transformado para CSV na mesma pasta do notebook
        pasta_notebook = obter_pasta_do_notebook()
        nome_arquivo_saida = "script_03_rotas_consolidado.csv"
        caminho_arquivo_saida = pasta_notebook / nome_arquivo_saida

        df_rotas_transformado.to_csv(
            caminho_arquivo_saida,
            index=False,
            encoding="utf-8-sig"
        )

        print(f"\nArquivo exportado com sucesso!")
        print(f"Caminho: {caminho_arquivo_saida}")


Base 'rotas' importada com sucesso!
Arquivo importado: d:\Matheus\OneDrive\Trabalho\00-UFG\Transporte escolar\Frente de indicadores\Dados SETE 2026\Dados dados sete consolidados\script_03_rotas.csv
Quantidade de linhas: 82887
Quantidade de colunas: 9

Base 'municípios' importada com sucesso!
Arquivo importado: d:\Matheus\OneDrive\Trabalho\00-UFG\Transporte escolar\Frente de indicadores\Dados SETE 2026\Dados dados sete consolidados\Lista de Municipios IBGE.csv
Quantidade de linhas: 5572
Quantidade de colunas: 13

Dados transformados com sucesso!
Quantidade de rotas sem cidade/estado encontradas: 0


,codigo_cidade,nome_cidade,estado,id_rota,da_porteira,da_mataburro,da_colchete,da_atoleiro,da_ponterustica,shape,tipo
0,5212501,Luziânia,Goiás,85313,N,N,N,N,N,NaN,1
1,3113305,Carangola,Minas Gerais,94777,N,N,N,N,N,NaN,1
2,1100098,Espigão D'Oeste,Rondônia,1,N,N,N,N,N,NaN,1
3,1100106,Guajará-Mirim,Rondônia,1,N,N,N,N,N,NaN,2
4,1100130,Machadinho D'Oeste,Rondônia,1,N,N,N,N,N,NaN,1
...,...,...,...,...,...,...,...,...,...,...,...
82882,2704708,Marechal Deodoro,Alagoas,173720,N,N,N,N,N,"{""type"":""FeatureCollection"",""features"":[{""type...",1
82883,4312708,Nonoai,Rio Grande do Sul,150978,N,N,N,N,N,"{""type"":""FeatureCollection"",""features"":[{""type...",1
82884,4125506,São José dos Pinhais,Paraná,173722,N,N,N,N,N,NaN,1
82885,3518404,Guaratinguetá,São Paulo,10,S,N,N,S,S,NaN,1



Arquivo exportado com sucesso!
Caminho: d:\Matheus\OneDrive\Trabalho\00-UFG\Transporte escolar\Frente de indicadores\Dados SETE 2026\Dados dados sete consolidados\script_03_rotas_consolidado.csv


In [2]:
# Bloco 2 - Preparar a base de rotas para consolidação

df_consolidacao = df_rotas_transformado.copy()

df_consolidacao["estado"] = df_consolidacao["estado"].fillna("Não identificado")

colunas_dificuldade = [
    "da_porteira",
    "da_mataburro",
    "da_colchete",
    "da_atoleiro",
    "da_ponterustica"
]

for coluna in colunas_dificuldade:
    df_consolidacao[coluna] = (
        df_consolidacao[coluna]
        .astype(str)
        .str.strip()
        .str.upper()
    )

df_consolidacao["tipo"] = pd.to_numeric(
    df_consolidacao["tipo"],
    errors="coerce"
).astype("Int64")

print("Base de rotas preparada para consolidação.")
display(df_consolidacao.head())

Base de rotas preparada para consolidação.


,codigo_cidade,nome_cidade,estado,id_rota,da_porteira,da_mataburro,da_colchete,da_atoleiro,da_ponterustica,shape,tipo
0,5212501,Luziânia,Goiás,85313,N,N,N,N,N,NaN,1
1,3113305,Carangola,Minas Gerais,94777,N,N,N,N,N,NaN,1
2,1100098,Espigão D'Oeste,Rondônia,1,N,N,N,N,N,NaN,1
3,1100106,Guajará-Mirim,Rondônia,1,N,N,N,N,N,NaN,2
4,1100130,Machadinho D'Oeste,Rondônia,1,N,N,N,N,N,NaN,1


In [3]:
# Bloco 3 - Função para consolidar dificuldades de acesso por estado

def consolidar_coluna_sim_nao_por_estado(df, coluna, nome_tabela):
    df_temp = df.copy()

    df_temp[coluna] = df_temp[coluna].astype(str).str.strip().str.upper()
    df_temp[f"tem_{coluna}"] = df_temp[coluna] == "S"

    tabela = (
        df_temp
        .groupby("estado")
        .agg(
            total_rotas=("id_rota", "size"),
            com=("tem_" + coluna, "sum"),
            sem=("tem_" + coluna, lambda x: (~x).sum())
        )
        .reset_index()
        .rename(
            columns={
                "com": f"com_{coluna}",
                "sem": f"sem_{coluna}"
            }
        )
        .sort_values("estado")
    )

    print(nome_tabela)
    display(tabela)

    return tabela

In [4]:
# Bloco 4 - Consolidar dificuldades de acesso por estado

tabela_porteira = consolidar_coluna_sim_nao_por_estado(
    df_consolidacao,
    "da_porteira",
    "Tabela consolidada de porteira por estado:"
)

tabela_mataburro = consolidar_coluna_sim_nao_por_estado(
    df_consolidacao,
    "da_mataburro",
    "Tabela consolidada de mata-burro por estado:"
)

tabela_colchete = consolidar_coluna_sim_nao_por_estado(
    df_consolidacao,
    "da_colchete",
    "Tabela consolidada de colchete por estado:"
)

tabela_atoleiro = consolidar_coluna_sim_nao_por_estado(
    df_consolidacao,
    "da_atoleiro",
    "Tabela consolidada de atoleiro por estado:"
)

tabela_ponterustica = consolidar_coluna_sim_nao_por_estado(
    df_consolidacao,
    "da_ponterustica",
    "Tabela consolidada de ponte rústica por estado:"
)

Tabela consolidada de porteira por estado:


,estado,total_rotas,com_da_porteira,sem_da_porteira
0,Acre,128,5,123
1,Alagoas,1345,31,1314
2,Amapá,47,0,47
3,Amazonas,332,0,332
4,Bahia,10080,415,9665
5,Ceará,4382,245,4137
6,Distrito Federal,5,0,5
7,Espírito Santo,1971,34,1937
8,Goiás,2491,646,1845
9,Maranhão,1968,187,1781


Tabela consolidada de mata-burro por estado:


,estado,total_rotas,com_da_mataburro,sem_da_mataburro
0,Acre,128,0,128
1,Alagoas,1345,15,1330
2,Amapá,47,0,47
3,Amazonas,332,0,332
4,Bahia,10080,361,9719
5,Ceará,4382,420,3962
6,Distrito Federal,5,0,5
7,Espírito Santo,1971,45,1926
8,Goiás,2491,705,1786
9,Maranhão,1968,212,1756


Tabela consolidada de colchete por estado:


,estado,total_rotas,com_da_colchete,sem_da_colchete
0,Acre,128,0,128
1,Alagoas,1345,6,1339
2,Amapá,47,0,47
3,Amazonas,332,0,332
4,Bahia,10080,154,9926
5,Ceará,4382,82,4300
6,Distrito Federal,5,0,5
7,Espírito Santo,1971,5,1966
8,Goiás,2491,411,2080
9,Maranhão,1968,169,1799


Tabela consolidada de atoleiro por estado:


,estado,total_rotas,com_da_atoleiro,sem_da_atoleiro
0,Acre,128,12,116
1,Alagoas,1345,135,1210
2,Amapá,47,13,34
3,Amazonas,332,30,302
4,Bahia,10080,895,9185
5,Ceará,4382,667,3715
6,Distrito Federal,5,0,5
7,Espírito Santo,1971,103,1868
8,Goiás,2491,517,1974
9,Maranhão,1968,561,1407


Tabela consolidada de ponte rústica por estado:


,estado,total_rotas,com_da_ponterustica,sem_da_ponterustica
0,Acre,128,9,119
1,Alagoas,1345,49,1296
2,Amapá,47,2,45
3,Amazonas,332,34,298
4,Bahia,10080,234,9846
5,Ceará,4382,110,4272
6,Distrito Federal,5,0,5
7,Espírito Santo,1971,32,1939
8,Goiás,2491,473,2018
9,Maranhão,1968,273,1695


In [5]:
# Bloco 5 - Consolidar tipo de rota por estado
# tipo: 1 - Rodoviária; 2 - Aquaviária; 3 - Mista

tabela_tipo_rota = (
    df_consolidacao
    .groupby("estado")
    .agg(
        total_rotas=("id_rota", "size"),
        rotas_rodoviarias=("tipo", lambda x: (x == 1).sum()),
        rotas_aquaviarias=("tipo", lambda x: (x == 2).sum()),
        rotas_mistas=("tipo", lambda x: (x == 3).sum()),
        rotas_tipo_nao_identificado=("tipo", lambda x: (~x.isin([1, 2, 3])).sum())
    )
    .reset_index()
    .sort_values("estado")
)

print("Tabela consolidada de tipo de rota por estado:")
display(tabela_tipo_rota)

Tabela consolidada de tipo de rota por estado:


,estado,total_rotas,rotas_rodoviarias,rotas_aquaviarias,rotas_mistas,rotas_tipo_nao_identificado
0,Acre,128,98,30,0,0
1,Alagoas,1345,1196,5,144,0
2,Amapá,47,36,11,0,0
3,Amazonas,332,124,208,0,0
4,Bahia,10080,9475,24,581,0
5,Ceará,4382,3908,1,473,0
6,Distrito Federal,5,5,0,0,0
7,Espírito Santo,1971,1848,1,122,0
8,Goiás,2491,2364,2,125,0
9,Maranhão,1968,1698,12,258,0


In [6]:
# Bloco 6 - Consolidar rotas com e sem georreferenciamento por estado
# Considera rota georreferenciada quando a coluna shape está preenchida

df_consolidacao["tem_georreferenciamento"] = df_consolidacao["shape"].notna()

tabela_georreferenciamento = (
    df_consolidacao
    .groupby("estado")
    .agg(
        total_rotas=("id_rota", "size"),
        rotas_geoferenciadas=("tem_georreferenciamento", "sum"),
        rotas_nao_georeferenciadas=("tem_georreferenciamento", lambda x: (~x).sum())
    )
    .reset_index()
    .sort_values("estado")
)

print("Tabela consolidada de rotas com e sem georreferenciamento por estado:")
display(tabela_georreferenciamento)

Tabela consolidada de rotas com e sem georreferenciamento por estado:


,estado,total_rotas,rotas_geoferenciadas,rotas_nao_georeferenciadas
0,Acre,128,24,104
1,Alagoas,1345,762,583
2,Amapá,47,1,46
3,Amazonas,332,191,141
4,Bahia,10080,3748,6332
5,Ceará,4382,586,3796
6,Distrito Federal,5,3,2
7,Espírito Santo,1971,622,1349
8,Goiás,2491,887,1604
9,Maranhão,1968,505,1463


In [7]:
# Bloco 7 - Consolidar todas as tabelas e exportar em Excel

from functools import reduce


def consolidar_tabelas_por_estado(tabelas):
    tabelas_preparadas = []

    for indice, tabela in enumerate(tabelas):
        tabela_temp = tabela.copy()

        if indice > 0 and "total_rotas" in tabela_temp.columns:
            tabela_temp = tabela_temp.drop(columns=["total_rotas"])

        tabelas_preparadas.append(tabela_temp)

    tabela_final = reduce(
        lambda esquerda, direita: esquerda.merge(
            direita,
            on="estado",
            how="outer"
        ),
        tabelas_preparadas
    )

    return tabela_final.sort_values("estado").reset_index(drop=True)


tabelas_para_consolidar = [
    tabela_porteira,
    tabela_mataburro,
    tabela_colchete,
    tabela_atoleiro,
    tabela_ponterustica,
    tabela_tipo_rota,
    tabela_georreferenciamento
]

tabela_consolidada_final = consolidar_tabelas_por_estado(tabelas_para_consolidar)

pasta_notebook = obter_pasta_do_notebook()
nome_arquivo_excel = "Consolidação 03 - script_03_rotas.xlsx"
caminho_arquivo_excel = pasta_notebook / nome_arquivo_excel

tabela_consolidada_final.to_excel(
    caminho_arquivo_excel,
    index=False,
    sheet_name="consolidado"
)

print("Tabela consolidada final exportada com sucesso!")
print(f"Arquivo Excel exportado: {caminho_arquivo_excel}")

display(tabela_consolidada_final)

Tabela consolidada final exportada com sucesso!
Arquivo Excel exportado: d:\Matheus\OneDrive\Trabalho\00-UFG\Transporte escolar\Frente de indicadores\Dados SETE 2026\Dados dados sete consolidados\Consolidação 03 - script_03_rotas.xlsx


,estado,total_rotas,com_da_porteira,sem_da_porteira,com_da_mataburro,sem_da_mataburro,com_da_colchete,sem_da_colchete,com_da_atoleiro,sem_da_atoleiro,com_da_ponterustica,sem_da_ponterustica,rotas_rodoviarias,rotas_aquaviarias,rotas_mistas,rotas_tipo_nao_identificado,rotas_geoferenciadas,rotas_nao_georeferenciadas
0,Acre,128,5,123,0,128,0,128,12,116,9,119,98,30,0,0,24,104
1,Alagoas,1345,31,1314,15,1330,6,1339,135,1210,49,1296,1196,5,144,0,762,583
2,Amapá,47,0,47,0,47,0,47,13,34,2,45,36,11,0,0,1,46
3,Amazonas,332,0,332,0,332,0,332,30,302,34,298,124,208,0,0,191,141
4,Bahia,10080,415,9665,361,9719,154,9926,895,9185,234,9846,9475,24,581,0,3748,6332
5,Ceará,4382,245,4137,420,3962,82,4300,667,3715,110,4272,3908,1,473,0,586,3796
6,Distrito Federal,5,0,5,0,5,0,5,0,5,0,5,5,0,0,0,3,2
7,Espírito Santo,1971,34,1937,45,1926,5,1966,103,1868,32,1939,1848,1,122,0,622,1349
8,Goiás,2491,646,1845,705,1786,411,2080,517,1974,473,2018,2364,2,125,0,887,1604
9,Maranhão,1968,187,1781,212,1756,169,1799,561,1407,273,1695,1698,12,258,0,505,1463
